# Strategy Validation

## CPCV Predictions

- **Purpose:** Generate CPCV splits, refit the primary and meta models within each split, and produce test-only strategy returns.
- **Settings:** `num_groups=6`; `num_test_groups=2`; `pct_embargo=0.01`; inner `cv=5`; `step_size=0.10`; `cost=0 bp/side`.
- **Data:** Use prepared development events and frozen model configurations to inspect split-level predictions and strategy returns.
- **Decision:** Purge and embargo overlapping information, keep every prediction out of sample within its split, and exclude the holdout partition.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone

PROJECT_ROOT = Path.cwd().resolve().parents[1]

from src.backtesting.strategy_validation import (
    combinatorial_purged_cross_validation,
    get_combinatorial_backtest_paths,
)
from src.backtesting.backtest_statistics import compute_strategy_returns
from src.backtesting.bet_sizing import get_signal
from src.modeling.purged_validation import PurgedKFold
from src.modeling.model_workflow import (
    build_meta_model_frame,
    build_primary_model_frame,
    generate_oof_predictions,
)

period = "2025-01-01_2025-12-31"
artifact_dir = PROJECT_ROOT / "data/model_artifact"
events = pd.read_parquet(PROJECT_ROOT / f"data/research_data/events/aapl_model_events_{period}.parquet").sort_values("event_start")
manifest = pd.read_parquet(artifact_dir / "split_manifest.parquet")
development_starts = manifest.loc[manifest["partition"].eq("development"), "event_start"]
development = build_primary_model_frame(events, development_starts)

primary_artifact = joblib.load(artifact_dir / "primary_model.joblib")
meta_artifact = joblib.load(artifact_dir / "meta_model.joblib")
primary_features = primary_artifact["feature_columns"]
meta_features = meta_artifact["feature_columns"]
information_sets = development["event_end"]

splits = combinatorial_purged_cross_validation(
    samples_info_sets=information_sets,
    num_groups=6,
    num_test_groups=2,
    pct_embargo=0.01,
)


In [2]:
split_predictions = []
for split in splits.itertuples(index=False):
    train = development.iloc[list(split.train_indices)].copy()
    test = development.iloc[list(split.test_indices)].copy()

    primary_train_estimator = clone(primary_artifact["estimator"])
    inner_cv = PurgedKFold(n_splits=5, t1=train["event_end"], pct_embargo=0.01)
    primary_train_oof = generate_oof_predictions(
        primary_train_estimator,
        train[primary_features],
        train["direction_label"].astype("int8"),
        train["sample_weight"],
        inner_cv,
        positive_label=1,
    )
    meta_train = build_meta_model_frame(
        train,
        primary_train_oof[["prediction", "probability", "prediction_source"]],
    )

    fitted_primary = clone(primary_artifact["estimator"]).fit(
        train[primary_features],
        train["direction_label"].astype("int8"),
        sample_weight=train["sample_weight"].to_numpy(),
    )
    primary_probability = fitted_primary.predict_proba(test[primary_features])[:, list(fitted_primary.classes_).index(1)]
    primary_side = fitted_primary.predict(test[primary_features]).astype("int8")

    test_meta = test.copy()
    test_meta["primary_side"] = primary_side
    test_meta["primary_probability"] = primary_probability
    test_meta["primary_confidence"] = np.maximum(primary_probability, 1.0 - primary_probability)

    fitted_meta = clone(meta_artifact["estimator"]).fit(
        meta_train[meta_features],
        meta_train["meta_label"].astype("int8"),
        sample_weight=meta_train["sample_weight"].to_numpy(),
    )
    meta_probability = fitted_meta.predict_proba(test_meta[meta_features])[:, list(fitted_meta.classes_).index(1)]
    meta_action = fitted_meta.predict(test_meta[meta_features]).astype("int8")

    result = test_meta[["event_end", "raw_return"]].copy()
    result["primary_side"] = primary_side
    result["meta_probability"] = meta_probability
    result["meta_action"] = meta_action
    result["bet_size"] = get_signal(
        events=result[["event_end"]].rename(columns={"event_end": "t1"}),
        step_size=0.10,
        prob=pd.Series(meta_probability, index=result.index),
        pred=pd.Series(meta_action, index=result.index),
        num_classes=2,
    )
    result = compute_strategy_returns(result, one_way_cost_bps=0.0)
    result["split_num"] = split.split_num
    result["observation_position"] = list(split.test_indices)
    split_predictions.append(result)

cpcv_predictions = pd.concat(split_predictions).sort_values(["split_num", "observation_position"])
display(cpcv_predictions.head())
display(cpcv_predictions.groupby("split_num")[["primary_only_net_return", "meta_filtered_net_return"]].sum())


,event_end,raw_return,primary_side,meta_probability,meta_action,bet_size,primary_only_position,meta_filtered_position,primary_only_gross_return,primary_only_entry_cost,primary_only_exit_cost,primary_only_total_cost,primary_only_net_return,meta_filtered_gross_return,meta_filtered_entry_cost,meta_filtered_exit_cost,meta_filtered_total_cost,meta_filtered_net_return,split_num,observation_position
event_start,,,,,,,,,,,,,,,,,,,,
2025-01-13 14:30:01.329809+00:00,2025-01-13 14:34:17.401727+00:00,-0.006778,1,0.466667,0,0.0,1.0,0.0,-0.006778,0.0,0.0,0.0,-0.006778,-0.000000,0.0,0.0,0.0,-0.000000,0,0
2025-01-16 14:30:01.488226+00:00,2025-01-16 14:34:43.996829+00:00,-0.004820,-1,0.425000,0,0.0,-1.0,0.0,0.004820,0.0,0.0,0.0,0.004820,-0.000000,0.0,0.0,0.0,-0.000000,0,1
2025-01-17 14:30:01.792073+00:00,2025-01-17 14:39:01.069235+00:00,-0.007032,-1,0.400000,0,0.0,-1.0,0.0,0.007032,0.0,0.0,0.0,0.007032,-0.000000,0.0,0.0,0.0,-0.000000,0,2
2025-01-17 17:20:53.298606+00:00,2025-01-21 14:30:00.854179+00:00,-0.025297,-1,0.391667,0,0.0,-1.0,0.0,0.025297,0.0,0.0,0.0,0.025297,-0.000000,0.0,0.0,0.0,-0.000000,0,3
2025-01-21 14:30:00.854179+00:00,2025-01-21 14:34:53.478136+00:00,-0.007594,1,0.558333,1,0.1,1.0,0.1,-0.007594,0.0,0.0,0.0,-0.007594,-0.000759,0.0,0.0,0.0,-0.000759,0,4


,primary_only_net_return,meta_filtered_net_return
split_num,,
0,0.137653,0.012765
1,0.064960,0.030071
2,0.131974,0.018457
3,0.099886,0.016120
4,-0.044989,-0.005865
5,0.282454,0.098290
6,0.005916,-0.001054
7,-0.104531,-0.002472
8,0.003825,-0.003533


## Backtest Paths

- **Purpose:** Assemble the split returns into complete development paths for PBO analysis.
- **Settings:** `num_groups=6`; `paths=5`; path values use `meta_filtered_net_return`.
- **Data:** Convert split-level CPCV predictions into an inspected split summary and five in-memory path-return series.
- **Decision:** Assign each development observation exactly once per path and reserve the paths for PBO analysis.

In [3]:
path_assignment = get_combinatorial_backtest_paths(splits, num_groups=6)
num_observations = len(development)
group_size = num_observations // 6
groups = [
    np.arange(group * group_size, (group + 1) * group_size)
    for group in range(5)
]
groups.append(np.arange(5 * group_size, num_observations))

path_returns = pd.DataFrame(index=development.index)
for path_name in path_assignment.columns:
    values = pd.Series(index=development.index, dtype="float64")
    for group, positions in enumerate(groups):
        split_num = int(path_assignment.loc[group, path_name])
        selected = cpcv_predictions[
            cpcv_predictions["split_num"].eq(split_num)
            & cpcv_predictions["observation_position"].isin(positions)
        ]
        values.loc[selected.index] = selected["meta_filtered_net_return"]
    path_returns[path_name] = values

split_summary = splits[["split_num", "train_groups", "test_groups"]].copy()
split_summary["train_events"] = splits["train_indices"].map(len)
split_summary["test_events"] = splits["test_indices"].map(len)

display(split_summary)
display(path_returns.describe().T)
display(path_assignment)


,split_num,train_groups,test_groups,train_events,test_events
0,0,"(2, 3, 4, 5)","(0, 1)",112,58
1,1,"(1, 3, 4, 5)","(0, 2)",113,58
2,2,"(1, 2, 4, 5)","(0, 3)",114,58
3,3,"(1, 2, 3, 5)","(0, 4)",114,58
4,4,"(1, 2, 3, 4)","(0, 5)",114,60
5,5,"(0, 3, 4, 5)","(1, 2)",116,58
6,6,"(0, 2, 4, 5)","(1, 3)",110,58
7,7,"(0, 2, 3, 5)","(1, 4)",110,58
8,8,"(0, 2, 3, 4)","(1, 5)",110,60
9,9,"(0, 1, 4, 5)","(2, 3)",115,58


,count,mean,std,min,25%,50%,75%,max
path_0,176.0,0.000090,0.003564,-0.010505,-0.000605,0.0,0.000764,0.018010
path_1,176.0,0.000697,0.003725,-0.007607,-0.000000,0.0,0.001349,0.022259
path_2,176.0,-0.000074,0.003032,-0.014155,-0.000896,0.0,0.000663,0.015639
path_3,176.0,0.000110,0.002483,-0.007091,-0.000000,0.0,0.000000,0.029117
path_4,176.0,0.000726,0.004965,-0.012679,-0.000691,-0.0,0.001232,0.029117


,path_0,path_1,path_2,path_3,path_4
group,,,,,
0,0,1,2,3,4
1,0,5,6,7,8
2,1,5,9,10,11
3,2,6,9,12,13
4,3,7,10,12,14
5,4,8,11,13,14
